# Red Circle Marking — Exploration Notebook

Comparing three approaches from the OCB Framework:
- **Crop**: object is cropped out of the image
- **Bounding Box**: rectangle drawn on the full original image
- **Red Circle**: ellipse drawn on the full original image (Shtedritski et al., ICCV 2023)

In [ ]:
import sys
sys.path.insert(0, "/workspaces/Object-Centric-Concept-Bottlenecks")

import torch
from PIL import Image, ImageDraw
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from utils import draw_box_on_image


## Load Image and Annotations

Using a Pascal VOC image — it comes with ready-made bounding box annotations in XML,
so we don't need to run any detection model.


In [ ]:
IMG_PATH = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/JPEGImages/2007_000032.jpg"
ANN_PATH = "/workspaces/Object-Centric-Concept-Bottlenecks/data/voc/VOCdevkit/VOC2012/Annotations/2007_000032.xml"

# Load image as tensor
image_pil = Image.open(IMG_PATH).convert("RGB")
image_tensor = TF.to_tensor(image_pil)  # shape: (3, H, W)

# Parse bounding boxes from VOC XML
tree = ET.parse(ANN_PATH)
root = tree.getroot()

boxes = []
labels = []
for obj in root.findall("object"):
    name = obj.find("name").text
    bb = obj.find("bndbox")
    x1 = int(bb.find("xmin").text)
    y1 = int(bb.find("ymin").text)
    x2 = int(bb.find("xmax").text)
    y2 = int(bb.find("ymax").text)
    boxes.append(torch.tensor([x1, y1, x2, y2]))
    labels.append(name)

print(f"Image size: {image_pil.size}")
print(f"Found {len(boxes)} objects: {labels}")
for i, b in enumerate(boxes):
    print(f"  {labels[i]}: {b.tolist()}")


## Original Image

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(image_pil)
plt.title("Original Image")
plt.axis("off")
plt.show()

## Approach 1: Bounding Box (existing implementation)

`draw_box_on_image` from utils.py draws a red rectangle around each detected object
on a copy of the full original image.

In [ ]:
fig, axes = plt.subplots(1, len(boxes), figsize=(15, 5))

for i, (box, label) in enumerate(zip(boxes, labels)):
    marked = draw_box_on_image(image_tensor, box)
    axes[i].imshow(marked)
    axes[i].set_title(f"{label}\n{box.tolist()}")
    axes[i].axis("off")

plt.suptitle("Bounding Box approach — one marked image per object", fontsize=13)
plt.tight_layout()
plt.show()

## Approach 2: Red Circle (new implementation)

Instead of a rectangle, we draw a red ellipse inscribed in the bounding box.
This is the approach from Shtedritski et al. (ICCV 2023).
The full image context is preserved — only the attention is directed via the circle.

In [ ]:
def draw_circle_on_image(image_tensor, box, color=(255, 0, 0), width=4):
    image_pil = TF.to_pil_image(image_tensor.cpu())
    draw = ImageDraw.Draw(image_pil) # Draw object
    x1, y1, x2, y2 = box.long()
    draw.ellipse([x1.item(), y1.item(), x2.item(), y2.item()], outline=color, width=width)
    return image_pil


fig, axes = plt.subplots(1, len(boxes), figsize=(15, 5))

for i, (box, label) in enumerate(zip(boxes, labels)):
    marked = draw_circle_on_image(image_tensor, box)
    axes[i].imshow(marked)
    axes[i].set_title(f"{label}\n{box.tolist()}")
    axes[i].axis("off")

plt.suptitle("Red Circle approach — one marked image per object", fontsize=13)
plt.tight_layout()
plt.show()

## Approach 3: Crop (original OCB implementation)

The object region is cropped out of the image — global context is lost.

In [ ]:
def crop_object(image_tensor, box):
    x1, y1, x2, y2 = box.long()
    cropped = image_tensor[:, y1:y2, x1:x2]
    return TF.to_pil_image(cropped.cpu())


fig, axes = plt.subplots(1, len(boxes), figsize=(15, 5))

for i, (box, label) in enumerate(zip(boxes, labels)):
    cropped = crop_object(image_tensor, box)
    axes[i].imshow(cropped)
    axes[i].set_title(f"{label}\n{box.tolist()}")
    axes[i].axis("off")

plt.suptitle("Crop approach — global context is lost", fontsize=13)
plt.tight_layout()
plt.show()

## Does CLIP "see" the Red Circle?

### Approach 1: Zero-Shot Classification

We give CLIP the marked image + text prompts and compare similarity scores.

In [ ]:
import open_clip

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
clip_model.eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")

print("CLIP model and tokenizer loaded successfully.")

In [ ]:
# Focus on the first object: large aeroplane
box = boxes[0]
label = labels[0]

# Prepare the three versions
crop_pil     = crop_object(image_tensor, box)
bbox_pil     = draw_box_on_image(image_tensor, box)
circle_pil   = draw_circle_on_image(image_tensor, box)

# Preprocess all three for CLIP (resize to 224x224, normalize)
crop_t   = clip_preprocess(crop_pil).unsqueeze(0)
bbox_t   = clip_preprocess(bbox_pil).unsqueeze(0)
circle_t = clip_preprocess(circle_pil).unsqueeze(0)

# Text prompts
prompts = [
    "a photo of an aeroplane",
    "a photo of a person",
    "a photo of a car",
    "a red circle around an aeroplane",
    "a red circle around a person",
]
text_tokens = tokenizer(prompts)

# Compute similarities
with torch.no_grad():
    crop_feat   = clip_model.encode_image(crop_t)
    bbox_feat   = clip_model.encode_image(bbox_t)
    circle_feat = clip_model.encode_image(circle_t)
    text_feat   = clip_model.encode_text(text_tokens)

    # Normalize
    crop_feat   = crop_feat   / crop_feat.norm(dim=-1, keepdim=True)
    bbox_feat   = bbox_feat   / bbox_feat.norm(dim=-1, keepdim=True)
    circle_feat = circle_feat / circle_feat.norm(dim=-1, keepdim=True)
    text_feat   = text_feat   / text_feat.norm(dim=-1, keepdim=True)

    crop_scores   = (crop_feat   @ text_feat.T).squeeze(0)
    bbox_scores   = (bbox_feat   @ text_feat.T).squeeze(0)
    circle_scores = (circle_feat @ text_feat.T).squeeze(0)

# Print results
print(f"{'Prompt':<40} {'Crop':>8} {'BBox':>8} {'Circle':>8}")
print("-" * 68)
for i, prompt in enumerate(prompts):
    print(f"{prompt:<40} {crop_scores[i].item():>8.3f} {bbox_scores[i].item():>8.3f} {circle_scores[i].item():>8.3f}")


## Approach 2: SpLiCE Concept Activations

SpLiCE decomposes the CLIP embedding into a sparse vector over a vocabulary of 10,000 words.
Each dimension = one concept word, value = activation strength.
We compare which concepts are activated for Crop vs. BBox vs. Red Circle.

In [ ]:
# Load cached embeddings directly
concepts = torch.load(
    "/root/.cache/splice/embeddings/open_clip:ViT-B-32_laion_10000_embeddings.pt",
    map_location="cpu"
)
image_mean = torch.load(
    "/root/.cache/splice/means/open_clip_ViT-B-32_image.pt",
    map_location="cpu"
)

# Load the correct LAION backbone
clip_backbone_laion, _, splice_preprocess_laion = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_backbone_laion.eval()

# Build SpLiCE with matching backbone
from splice.model import SPLICE
splice_model = SPLICE(
    image_mean=image_mean,
    dictionary=concepts,
    clip=clip_backbone_laion,
    device="cpu",
    l1_penalty=0.25,
    return_weights=True,
)
splice_model.eval()
print("SpLiCE with correct backbone loaded")


## SpLiCE Concept Activations: Crop vs. Bounding Box vs. Red Circle

In [ ]:
import splice

vocab = splice.get_vocabulary("laion", vocabulary_size=10000)
print(f"Vocabulary size: {len(vocab)}")
print(f"First 10 words: {vocab[:10]}")
print(f"Last 10 words: {vocab[-10:]}")

In [ ]:
def get_top_concepts(image_pil, splice_model, splice_preprocess, vocab, top_k=10):
    img_t = splice_preprocess(image_pil).unsqueeze(0)
    with torch.no_grad():
        activations, _ = splice_model.encode_image(img_t)
    activations = activations.squeeze(0)
    top_indices = activations.argsort(descending=True)[:top_k]
    return [(vocab[i], activations[i].item()) for i in top_indices]


crop_concepts   = get_top_concepts(crop_pil,   splice_model, clip_preprocess, vocab)
bbox_concepts   = get_top_concepts(bbox_pil,   splice_model, clip_preprocess, vocab)
circle_concepts = get_top_concepts(circle_pil, splice_model, clip_preprocess, vocab)

print(f"{'Crop':<30} {'BBox':<30} {'Circle':<30}")
print("-" * 90)
for i in range(10):
    c  = f"{crop_concepts[i][0]} ({crop_concepts[i][1]:.3f})"
    b  = f"{bbox_concepts[i][0]} ({bbox_concepts[i][1]:.3f})"
    r  = f"{circle_concepts[i][0]} ({circle_concepts[i][1]:.3f})"
    print(f"{c:<30} {b:<30} {r:<30}")


## Concept Activations for All Objects

In [ ]:
for idx, (box, label) in enumerate(zip(boxes, labels)):
    crop_pil_i   = crop_object(image_tensor, box)
    bbox_pil_i   = draw_box_on_image(image_tensor, box)
    circle_pil_i = draw_circle_on_image(image_tensor, box)

    crop_c   = get_top_concepts(crop_pil_i,   splice_model, splice_preprocess_laion, vocab)
    bbox_c   = get_top_concepts(bbox_pil_i,   splice_model, splice_preprocess_laion, vocab)
    circle_c = get_top_concepts(circle_pil_i, splice_model, splice_preprocess_laion, vocab)

    print(f"\n{'='*90}")
    print(f"Object {idx+1}: {label}  |  box: {box.tolist()}")
    print(f"{'='*90}")
    print(f"{'Crop':<30} {'BBox':<30} {'Circle':<30}")
    print("-" * 90)
    for i in range(10):
        c = f"{crop_c[i][0]} ({crop_c[i][1]:.3f})"
        b = f"{bbox_c[i][0]} ({bbox_c[i][1]:.3f})"
        r = f"{circle_c[i][0]} ({circle_c[i][1]:.3f})"
        print(f"{c:<30} {b:<30} {r:<30}")

## Aggregation: Full Image + All Objects

Simulating the full OCB pipeline: encode full image + all objects, then aggregate.
Comparing aggregation methods (sum, max) across all three marking approaches.

In [ ]:
def get_splice_embedding(image_pil, splice_model, preprocess):
    img_t = preprocess(image_pil).unsqueeze(0)
    with torch.no_grad():
        emb, _ = splice_model.encode_image(img_t)
    return emb.squeeze(0)

# Full image encoding (no marking)
full_img_emb = get_splice_embedding(image_pil, splice_model, splice_preprocess_laion)

# Object encodings for each approach
crop_embs   = [get_splice_embedding(crop_object(image_tensor, b),          splice_model, splice_preprocess_laion) for b in boxes]
bbox_embs   = [get_splice_embedding(draw_box_on_image(image_tensor, b),    splice_model, splice_preprocess_laion) for b in boxes]
circle_embs = [get_splice_embedding(draw_circle_on_image(image_tensor, b), splice_model, splice_preprocess_laion) for b in boxes]

# Stack to (num_objects, embedding_dim)
crop_stack   = torch.stack(crop_embs)
bbox_stack   = torch.stack(bbox_embs)
circle_stack = torch.stack(circle_embs)

print(f"Full image embedding shape: {full_img_emb.shape}")
print(f"Object embeddings shape:    {crop_stack.shape}")


In [ ]:
def get_top_concepts_from_embedding(embedding, vocab, top_k=10):
    top_indices = embedding.argsort(descending=True)[:top_k]
    return [(vocab[i], embedding[i].item()) for i in top_indices]

crop_sum   = full_img_emb + crop_stack.sum(dim=0)
bbox_sum   = full_img_emb + bbox_stack.sum(dim=0)
circle_sum = full_img_emb + circle_stack.sum(dim=0)

crop_top   = get_top_concepts_from_embedding(crop_sum,   vocab)
bbox_top   = get_top_concepts_from_embedding(bbox_sum,   vocab)
circle_top = get_top_concepts_from_embedding(circle_sum, vocab)

print(f"{'Crop + Sum':<30} {'BBox + Sum':<30} {'Circle + Sum':<30}")
print("-" * 90)
for i in range(10):
    c = f"{crop_top[i][0]} ({crop_top[i][1]:.3f})"
    b = f"{bbox_top[i][0]} ({bbox_top[i][1]:.3f})"
    r = f"{circle_top[i][0]} ({circle_top[i][1]:.3f})"
    print(f"{c:<30} {b:<30} {r:<30}")


In [ ]:
print("How airport (0.830) is computed in Circle + Sum:\n")

airport_idx = vocab.index("airport")

full_img_val = full_img_emb[airport_idx].item()
print(f"Full image:     {full_img_val:.4f}")

for i, (emb, label) in enumerate(zip(circle_stack, labels)):
    val = emb[airport_idx].item()
    print(f"Object {i+1} ({label}): {val:.4f}")

print(f"\nSum of objects: {circle_stack[:, airport_idx].sum().item():.4f}")
print(f"Full + Sum:     {circle_sum[airport_idx].item():.4f}")

In [ ]:
from utils import obj_enc_aggregation

# Shape muss (batch, num_objects, embedding_dim) sein
full_img_batch = full_img_emb.unsqueeze(0)          # (1, 10000)
crop_batch     = crop_stack.unsqueeze(0)             # (1, 4, 10000)
bbox_batch     = bbox_stack.unsqueeze(0)             # (1, 4, 10000)
circle_batch   = circle_stack.unsqueeze(0)           # (1, 4, 10000)

crop_count   = obj_enc_aggregation("count", crop_batch,   full_img_batch).squeeze(0)
bbox_count   = obj_enc_aggregation("count", bbox_batch,   full_img_batch).squeeze(0)
circle_count = obj_enc_aggregation("count", circle_batch, full_img_batch).squeeze(0)

def top_count_concepts(count_emb, vocab, top_k=10):
    idx = count_emb.argsort(descending=True)[:top_k]
    return [(vocab[i], int(count_emb[i].item())) for i in idx]

crop_top   = top_count_concepts(crop_count,   vocab)
bbox_top   = top_count_concepts(bbox_count,   vocab)
circle_top = top_count_concepts(circle_count, vocab)

print(f"{'Crop + Count':<30} {'BBox + Count':<30} {'Circle + Count':<30}")
print("(value = how many sources activated this concept, max=5)")
print("-" * 90)
for i in range(10):
    c = f"{crop_top[i][0]} ({crop_top[i][1]})"
    b = f"{bbox_top[i][0]} ({bbox_top[i][1]})"
    r = f"{circle_top[i][0]} ({circle_top[i][1]})"
    print(f"{c:<30} {b:<30} {r:<30}")